In [1]:
from datasets import load_dataset

dataset = load_dataset(
    "Helsinki-NLP/opus-100",
    "en-zh"
)

train = dataset["train"]

for batch in train.iter(batch_size=8):
    pairs = batch["translation"]
    en_texts = [pair["en"] for pair in pairs]
    zh_texts = [pair["zh"] for pair in pairs]

    # 先看第一批的第一对句子，删掉 break 就会遍历所有批次
    print(en_texts)
    print(zh_texts)
    break

['Sixty-first session', 'I took some medicine for my mu for my mu my muscular for my muscular...', "It's a challenge. God is challenging you. He's calling you a chump.", 'Oh, baby.', '- Lucinda?', 'Introduction', 'Eric Topol: The wireless future of medicine', 'Damn it, Vaughn!']
['第六十一届会议', '减轻酸... 酸痛的药 减轻酸痛的药', '上帝在挑战你，他说你是笨蛋', '.. 寶貝', '- 盧辛達？', '一. 导言', 'Eric Topol：未来医疗的无线化', '妈的, 沃恩!']


In [2]:
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

tokenizer_dir = snapshot_download(
    repo_id="facebook/nllb-200-distilled-600M",
    allow_patterns=[
        "config.json",
        "tokenizer_config.json",
        "tokenizer.json",
        "special_tokens_map.json",
        "sentencepiece.bpe.model",
    ],
)

tokenizer = AutoTokenizer.from_pretrained(
    tokenizer_dir,
    local_files_only=True,
    src_lang="eng_Latn",
    tgt_lang="zho_Hans",
)

pair = dataset["train"][0]["translation"]
encoded = tokenizer(pair["en"], text_target=pair["zh"])
print("英文 token IDs", encoded["input_ids"])
print("中文 token IDs", encoded["labels"])

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

英文 token IDs [256047, 1769, 3993, 248078, 248105, 6964, 148, 107742, 2]
中文 token IDs [256200, 23497, 164186, 248833, 80220, 2]


In [3]:
from torch.utils.data import DataLoader
from transformers import DataCollatorForSeq2Seq

def tokenize_batch(batch):
    pairs = batch["translation"]
    return tokenizer(
        [pair["en"] for pair in pairs],
        text_target=[pair["zh"] for pair in pairs],
        truncation=True,
        max_length=128,
    )

tokenized_train = train.map(
    tokenize_batch,
    batched=True,
    remove_columns=train.column_names,
)

loader = DataLoader(
    tokenized_train,
    batch_size=32,
    shuffle=True,
    collate_fn=DataCollatorForSeq2Seq(tokenizer=tokenizer),
)

batch = next(iter(loader))
print(batch["input_ids"].shape)  # [批大小, 补齐后的英文长度]
print(batch["labels"].shape)     # [批大小, 补齐后的中文长度]

Map:   0%|          | 0/1000000 [00:00<?, ? examples/s]

torch.Size([32, 81])
torch.Size([32, 81])
